# Notebook 15 — Best CZ Gate Extraction and Paper Figure

This notebook extracts a single **best shaped-adiabatic CZ candidate** and presents it in a compact, paper-style form.

It does four things:

1. scans a refined local parameter region around the shaped-protocol optimum,
2. identifies the best candidate using the composite score,
3. reports the final gate metrics and matrix error against ideal CZ,
4. generates a compact figure set suitable for a paper, README, or project note.

This notebook is the final “best gate” capstone for the repo.


In [ ]:
# Ensure dependencies are installed before imports
import importlib, subprocess, sys

def ensure(pkg):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

for pkg in ['qutip', 'numpy', 'matplotlib']:
    ensure(pkg)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qutip import basis, qeye, tensor, sigmax, sesolve


## Basis states and operators

In [ ]:
g = basis(2, 0)
r = basis(2, 1)
I = qeye(2)

sx = sigmax()
n = r * r.dag()

gg = tensor(g, g)
gr = tensor(g, r)
rg = tensor(r, g)
rr = tensor(r, r)

basis_states = [gg, gr, rg, rr]
basis_labels = ['|gg>', '|gr>', '|rg>', '|rr>']

sx1 = tensor(sx, I)
sx2 = tensor(I, sx)
n1 = tensor(n, I)
n2 = tensor(I, n)

U_cz = np.diag([1, 1, 1, -1]).astype(complex)


## Shaped schedules and Hamiltonian

In [ ]:
def omega_shaped(t, T, omega_max, p):
    s = t / T
    base = 16.0 * (s * (1.0 - s)) ** 2
    return omega_max * np.maximum(base, 0.0) ** p

def delta_shaped(t, T, V, alpha, delta0, q):
    s = t / T
    shaped = s ** q
    return delta0 + alpha * V * 0.5 * (1.0 - np.cos(np.pi * shaped))

H_omega = 0.5 * (sx1 + sx2)
H_delta = -(n1 + n2)
H_V = n1 * n2

def build_time_dependent_hamiltonian(T, omega_max, alpha, V, delta0, p, q):
    def coeff_omega(t, args=None):
        return omega_shaped(t, T=T, omega_max=omega_max, p=p)

    def coeff_delta(t, args=None):
        return delta_shaped(t, T=T, V=V, alpha=alpha, delta0=delta0, q=q)

    return [
        [H_omega, coeff_omega],
        [H_delta, coeff_delta],
        [H_V, lambda t, args=None: V],
    ]


## Effective-unitary helpers

In [ ]:
def run_shaped_evolution(psi0, T, omega_max, alpha, V, delta0, p, q, n_steps=500):
    H = build_time_dependent_hamiltonian(
        T=T, omega_max=omega_max, alpha=alpha, V=V, delta0=delta0, p=p, q=q
    )
    times = np.linspace(0.0, T, n_steps)
    result = sesolve(H, psi0, times)
    return result.states[-1]

def state_to_column(psi):
    return np.array([basis_state.overlap(psi) for basis_state in basis_states], dtype=complex)

def build_effective_unitary(T, omega_max, alpha, V, delta0, p, q, n_steps=500):
    columns = []
    for psi0 in basis_states:
        psi_final = run_shaped_evolution(
            psi0, T=T, omega_max=omega_max, alpha=alpha, V=V,
            delta0=delta0, p=p, q=q, n_steps=n_steps
        )
        columns.append(state_to_column(psi_final))
    return np.column_stack(columns)


## Gate metrics

In [ ]:
def process_fidelity(U_eff, U_target=U_cz):
    d = U_target.shape[0]
    return float(np.abs(np.trace(np.conjugate(U_target.T) @ U_eff)) ** 2 / (d ** 2))

def wrapped_phase(x):
    return np.angle(np.exp(1j * x))

def diagonal_phases(U_eff):
    return np.angle(np.diag(U_eff))

def entangling_phase(U_eff):
    phi = diagonal_phases(U_eff)
    return wrapped_phase(phi[0] + phi[3] - phi[1] - phi[2])

def phase_corrected_target(U_eff):
    phi_ent = entangling_phase(U_eff)
    return np.diag([1, 1, 1, np.exp(1j * phi_ent)]).astype(complex)

def phase_corrected_fidelity(U_eff):
    return process_fidelity(U_eff, U_target=phase_corrected_target(U_eff))

def leakage_metric(U_eff):
    offdiag = U_eff.copy()
    np.fill_diagonal(offdiag, 0)
    return float(np.linalg.norm(offdiag))

def composite_score(U_eff):
    return (
        0.5 * phase_corrected_fidelity(U_eff)
        + 0.3 * process_fidelity(U_eff)
        - 0.2 * leakage_metric(U_eff)
    )

def matrix_error_to_cz(U_eff):
    return float(np.linalg.norm(U_eff - U_cz))

def matrix_error_to_phase_corrected(U_eff):
    return float(np.linalg.norm(U_eff - phase_corrected_target(U_eff)))


## Refined local search around the shaped optimum

In [ ]:
# Center near the best shaped region seen in Notebook 13/14.5
V = 4.0
T_vals = np.linspace(22.0, 31.0, 19)
alpha_vals = np.linspace(0.10, 0.24, 15)
omega_vals = np.linspace(1.00, 1.20, 11)
delta0_vals = np.linspace(0.70, 1.20, 11)
p_vals = np.linspace(0.80, 1.20, 9)
q_vals = np.linspace(0.60, 0.90, 7)

best = None
best_score = -1e9

for T in T_vals:
    for alpha in alpha_vals:
        for omega_max in omega_vals:
            for delta0 in delta0_vals:
                for p in p_vals:
                    for q in q_vals:
                        U = build_effective_unitary(
                            T=T, omega_max=omega_max, alpha=alpha, V=V,
                            delta0=delta0, p=p, q=q, n_steps=360
                        )
                        score = composite_score(U)
                        if score > best_score:
                            best_score = score
                            best = {
                                'T': float(T),
                                'alpha': float(alpha),
                                'omega_max': float(omega_max),
                                'delta0': float(delta0),
                                'p': float(p),
                                'q': float(q),
                                'score': float(score),
                                'U': U,
                            }

best


## Best gate report

In [ ]:
U_best = best['U']

report = {
    'T': best['T'],
    'alpha': best['alpha'],
    'omega_max': best['omega_max'],
    'delta0': best['delta0'],
    'p': best['p'],
    'q': best['q'],
    'raw_fidelity': process_fidelity(U_best),
    'phase_corrected_fidelity': phase_corrected_fidelity(U_best),
    'entangling_phase_over_pi': entangling_phase(U_best) / np.pi,
    'leakage': leakage_metric(U_best),
    'composite_score': composite_score(U_best),
    'matrix_error_to_CZ': matrix_error_to_cz(U_best),
    'matrix_error_to_phase_corrected_target': matrix_error_to_phase_corrected(U_best),
}

for k, v in report.items():
    print(f"{k}: {v:.6f}")


## Publication-style figure 1 — best effective unitary

In [ ]:
plt.figure(figsize=(5, 4))
im = plt.imshow(np.abs(U_best), origin='lower', aspect='equal')
plt.title('Best extracted shaped CZ candidate')
plt.xticks(range(4), basis_labels)
plt.yticks(range(4), basis_labels)
plt.colorbar(im, label='magnitude')
plt.tight_layout()
plt.show()


## Publication-style figure 2 — diagonal and off-diagonal structure

In [ ]:
diag_vals = np.abs(np.diag(U_best))
offdiag = U_best.copy()
np.fill_diagonal(offdiag, 0.0)
offdiag_norms = np.sum(np.abs(offdiag), axis=0)

plt.figure(figsize=(7, 4.5))
x = np.arange(4)
w = 0.35
plt.bar(x - w/2, diag_vals, width=w, label='|diagonal element|')
plt.bar(x + w/2, offdiag_norms, width=w, label='sum |off-diagonal column|')
plt.xticks(x, basis_labels)
plt.ylabel('Magnitude')
plt.title('Best gate: diagonal dominance vs off-diagonal spill')
plt.legend()
plt.tight_layout()
plt.show()


## Publication-style figure 3 — local score map around the best gate

In [ ]:
T_local = np.linspace(max(20.0, best['T']-4.0), min(33.0, best['T']+4.0), 24)
alpha_local = np.linspace(max(0.08, best['alpha']-0.08), min(0.35, best['alpha']+0.08), 20)

score_map = np.zeros((len(alpha_local), len(T_local)))
pc_map = np.zeros((len(alpha_local), len(T_local)))

for i, alpha in enumerate(alpha_local):
    for j, T in enumerate(T_local):
        U = build_effective_unitary(
            T=T, omega_max=best['omega_max'], alpha=alpha, V=V,
            delta0=best['delta0'], p=best['p'], q=best['q'], n_steps=320
        )
        score_map[i, j] = composite_score(U)
        pc_map[i, j] = phase_corrected_fidelity(U)

plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(score_map, origin='lower', aspect='auto',
                extent=[T_local[0], T_local[-1], alpha_local[0], alpha_local[-1]])
plt.contour(T_local, alpha_local, score_map, colors='white', linewidths=0.4)
plt.scatter([best['T']], [best['alpha']], marker='x', s=80)
plt.xlabel('Total gate time T')
plt.ylabel(r'Detuning scale factor $\alpha$')
plt.title('Local score basin around the best extracted gate')
plt.colorbar(im, label='composite score')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(pc_map, origin='lower', aspect='auto',
                extent=[T_local[0], T_local[-1], alpha_local[0], alpha_local[-1]])
plt.contour(T_local, alpha_local, pc_map, colors='white', linewidths=0.4)
plt.scatter([best['T']], [best['alpha']], marker='x', s=80)
plt.xlabel('Total gate time T')
plt.ylabel(r'Detuning scale factor $\alpha$')
plt.title('Local phase-corrected fidelity basin')
plt.colorbar(im, label='phase-corrected fidelity')
plt.tight_layout()
plt.show()


## Compact summary text block

In [ ]:
summary_text = f'''
Best extracted shaped-adiabatic CZ candidate

T      = {report["T"]:.4f}
alpha  = {report["alpha"]:.4f}
Omega  = {report["omega_max"]:.4f}
Delta0 = {report["delta0"]:.4f}
p      = {report["p"]:.4f}
q      = {report["q"]:.4f}

raw fidelity                 = {report["raw_fidelity"]:.4f}
phase-corrected fidelity     = {report["phase_corrected_fidelity"]:.4f}
entangling phase / pi        = {report["entangling_phase_over_pi"]:.4f}
leakage                      = {report["leakage"]:.4f}
composite score              = {report["composite_score"]:.4f}
matrix error to ideal CZ     = {report["matrix_error_to_CZ"]:.4f}
matrix error to phase target = {report["matrix_error_to_phase_corrected_target"]:.4f}
'''
print(summary_text)


## Final conclusion

This notebook extracts a single best gate candidate from the shaped-adiabatic family.

The result is now compact enough to use directly in:
- a README,
- a short project note,
- a paper draft,
- or an outreach figure.

The full repo conclusion is now explicit:

**shaped adiabatic control gives the cleanest gate-like structure, the strongest coherent performance, and the best tradeoff between phase alignment and leakage in this modeled system.**
